# Dashboard Extracts — Power BI Source Layer
Group 03 · Digital Payments Settlement Latency & Merchant Churn Analytics.

Runs against `payments_analytics.db` (located relative to this notebook — project root or `dashboard/`)
and writes **7 tidy CSV extracts** to `dashboard/extracts/` — the import layer for the Power BI model
(see `docs/dashboard_spec_powerbi.md`). All aggregation is pushed down to SQLite; CSVs are
intentionally small, dashboards must not re-derive metrics from raw facts. Re-run this notebook after
any DB refresh.

In [1]:
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text

# Locate the project root (notebook may be run from the project root or from dashboard/)
ROOT = Path.cwd()
while not (ROOT / "payments_analytics.db").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DB_PATH = ROOT / "payments_analytics.db"
assert DB_PATH.exists(), f"payments_analytics.db not found above {Path.cwd()}"
OUT_DIR = ROOT / "dashboard" / "extracts"
OUT_DIR.mkdir(parents=True, exist_ok=True)
engine = create_engine(f"sqlite:///{DB_PATH}")
SR = "AVG(CASE WHEN t.status = 'SUCCESS' THEN 1.0 ELSE 0 END) * 100"


def run_sql(sql: str) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)


def save(df: pd.DataFrame, name: str) -> None:
    df.to_csv(OUT_DIR / name, index=False)
    print(f"{name:<28s} {len(df):>7,} rows")


print("Extracts ->", OUT_DIR.resolve())

Extracts -> E:\26-28\MBA\Trim_1\Programming_for_analytics\Capstone Project\dashboard\extracts


### 1 · `monthly_overview.csv` — month grain
Volume, success rate and latency per month (cut at 2024-12). Powers trend cards on the Gateway and
Executive pages.

In [2]:
monthly = run_sql(f"""
    SELECT SUBSTR(t.txn_timestamp, 1, 7)            AS month,
           COUNT(*)                                 AS txns,
           {SR}                                     AS success_rate_pct,
           AVG(t.latency_ms)                        AS avg_latency_ms,
           SUM(CASE WHEN t.latency_ms >= 10000 THEN 1 ELSE 0 END) AS latency_spikes
    FROM fact_payment_transactions t
    WHERE t.txn_timestamp < '2025-01-01'
    GROUP BY 1 ORDER BY 1
""")
save(monthly, "monthly_overview.csv")

monthly_overview.csv              18 rows


### 2 · `bank_performance.csv` — issuing-bank grain
The gateway health table: volume, success/timeout rates, avg + p95 latency, spike share, PSU/private
tier. Powers the bank league table and tier comparison.

In [3]:
banks = run_sql(f"""
    SELECT b.bank_name,
           b.tier,
           COUNT(*)                                                      AS txns,
           {SR}                                                          AS success_rate_pct,
           AVG(CASE WHEN t.status = 'TIMEOUT' THEN 1.0 ELSE 0 END) * 100 AS timeout_pct,
           AVG(t.latency_ms)                                             AS avg_latency_ms,
           AVG(CASE WHEN t.latency_ms >= 10000 THEN 1.0 ELSE 0 END) * 100 AS spike_pct
    FROM fact_payment_transactions t JOIN dim_issuing_banks b USING (bank_id)
    GROUP BY 1, 2
""")
p95 = run_sql("""
    WITH ranked AS (
        SELECT b.bank_name, t.latency_ms,
               ROW_NUMBER() OVER (PARTITION BY b.bank_name ORDER BY t.latency_ms) AS rn,
               COUNT(*)     OVER (PARTITION BY b.bank_name)                       AS n
        FROM fact_payment_transactions t JOIN dim_issuing_banks b USING (bank_id)
    )
    SELECT bank_name, MAX(latency_ms) AS p95_latency_ms
    FROM ranked WHERE rn = CAST(0.95 * n AS INTEGER) + 1
    GROUP BY bank_name
""")
banks = banks.merge(p95, on="bank_name")
save(banks, "bank_performance.csv")

bank_performance.csv              12 rows


### 3 · `error_pareto.csv` — error-code grain
Failure mix with the infrastructure/user-side split and cumulative share for Pareto visuals.

In [4]:
pareto = run_sql("""
    SELECT error_code,
           CASE WHEN error_code IN ('ERR_BANK_TIMEOUT', 'ERR_SWITCH_UNAVAILABLE', 'ERR_NPCI_DEGRADED')
                THEN 'Infrastructure' ELSE 'User-side' END AS fault_domain,
           COUNT(*) AS failures
    FROM fact_payment_transactions
    WHERE status <> 'SUCCESS'
    GROUP BY 1, 2 ORDER BY failures DESC
""")
pareto["share_pct"] = pareto["failures"] / pareto["failures"].sum() * 100
pareto["cumulative_pct"] = pareto["share_pct"].cumsum()
save(pareto, "error_pareto.csv")

error_pareto.csv                   5 rows


### 4 · `settlement_performance.csv` — cycle × batch-month grain
SLA breach rate and delay severity per contracted cycle per batch month (long grain: cycle bars and
monthly breach trend both come from this one extract). Months cut at 2024-12 (2025-01 stub excluded).

In [5]:
settlements = run_sql("""
    SELECT m.settlement_cycle_sla                                            AS sla_cycle,
           SUBSTR(s.settlement_batch_date, 1, 7)                             AS batch_month,
           COUNT(*)                                                          AS batches,
           SUM(s.sla_breach_flag)                                            AS breaches,
           100.0 * SUM(s.sla_breach_flag) / COUNT(*)                         AS breach_pct,
           AVG(s.settlement_delay_days)                                      AS avg_delay_days,
           AVG(CASE WHEN s.sla_breach_flag = 1 THEN s.settlement_delay_days END) AS avg_delay_if_breach_days
    FROM fact_merchant_settlements s JOIN dim_merchants m USING (merchant_id)
    WHERE s.settlement_batch_date <= '2024-12-31'
    GROUP BY 1, 2
""")
save(settlements, "settlement_performance.csv")

settlement_performance.csv        72 rows


### 5 · `merchant_exposure.csv` — merchant grain
One row per settled merchant: tier, category, breach bucket, volume/revenue run-rate and churn flags.
Powers the churn page (scatter/bucket visuals), slicers, and the protected-volume card.

In [6]:
exposure = run_sql("""
    SELECT m.merchant_id,
           m.merchant_tier                                   AS tier,
           m.mcc_category                                    AS category,
           m.settlement_cycle_sla                            AS sla_cycle,
           m.onboarding_date,
           m.churn_date,
           m.churn_reason,
           (m.churn_date IS NOT NULL)                        AS is_churned,
           COUNT(*)                                          AS batches,
           AVG(s.sla_breach_flag * 1.0)                      AS breach_rate,
           CASE
               WHEN AVG(s.sla_breach_flag * 1.0) = 0     THEN '0%'
               WHEN AVG(s.sla_breach_flag * 1.0) <= 0.10 THEN '0-10%'
               WHEN AVG(s.sla_breach_flag * 1.0) <= 0.25 THEN '10-25%'
               ELSE '>25%' END                           AS breach_bucket,
           SUM(s.gross_volume_inr)                           AS gpv_inr,
           SUM(s.net_mdr_deducted_inr)                       AS mdr_inr
    FROM fact_merchant_settlements s JOIN dim_merchants m USING (merchant_id)
    GROUP BY 1, 2, 3, 4, 5, 6, 7, 8
""")
save(exposure, "merchant_exposure.csv")

merchant_exposure.csv          2,517 rows


### 6 · `routing_ab.csv` — route × tier × channel grain
The A/B engine evidence at segment grain: success rate and latency per routing mode — powers the
smart-routing lift visuals (overall + by tier + by channel).

In [7]:
routing = run_sql(f"""
    SELECT t.is_routed_via_dynamic_sla                       AS routed,
           m.merchant_tier                                   AS tier,
           pm.payment_channel                                AS channel,
           COUNT(*)                                          AS txns,
           {SR}                                              AS success_rate_pct,
           AVG(t.latency_ms)                                 AS avg_latency_ms
    FROM fact_payment_transactions t
    JOIN dim_merchants m USING (merchant_id)
    JOIN dim_payment_methods pm USING (payment_method_id)
    GROUP BY 1, 2, 3
""")
save(routing, "routing_ab.csv")

routing_ab.csv                    30 rows


### 7 · `channel_economics.csv` — tier × channel grain
Successful GMV and potential MDR (volume × contracted `mdr_rate_pct`) per tier × channel — powers the
revenue-mix matrix and the UPI-vs-cards monetization story.

In [8]:
economics = run_sql("""
    SELECT m.merchant_tier                                   AS tier,
           pm.payment_channel                                AS channel,
           SUM(t.amount_inr)                                 AS gmv_success_inr,
           SUM(t.amount_inr * pm.mdr_rate_pct / 100)         AS mdr_potential_inr,
           MIN(pm.mdr_rate_pct)                              AS mdr_rate_min,
           MAX(pm.mdr_rate_pct)                              AS mdr_rate_max
    FROM fact_payment_transactions t
    JOIN dim_payment_methods pm USING (payment_method_id)
    JOIN dim_merchants m USING (merchant_id)
    WHERE t.status = 'SUCCESS'
    GROUP BY 1, 2
""")
save(economics, "channel_economics.csv")

channel_economics.csv             15 rows


### Validation
Row counts, file sizes and a quick aggregate cross-check (overall SR from the monthly extract must
equal the transaction-weighted mean; breach share from the settlement extract must match ~28%).

In [9]:
files = sorted(OUT_DIR.glob("*.csv"))
summary = pd.DataFrame(
    [{"file": f.name, "rows": sum(1 for _ in open(f, encoding="utf-8")) - 1,
      "bytes": f.stat().st_size} for f in files])
display(summary)

chk_sr = (monthly["txns"] * monthly["success_rate_pct"]).sum() / monthly["txns"].sum()
chk_breach = settlements["breaches"].sum() / settlements["batches"].sum() * 100
assert abs(chk_sr - 83.69) < 0.05, f"SR cross-check failed: {chk_sr:.2f}"
assert abs(chk_breach - 28.4) < 0.3, f"Breach cross-check failed: {chk_breach:.2f}"
print(f"Cross-checks OK — txn-weighted SR {chk_sr:.2f}% | settlement breach {chk_breach:.2f}%")

,file,rows,bytes
0,bank_performance.csv,12,1422
1,channel_economics.csv,15,831
2,error_pareto.csv,5,423
3,merchant_exposure.csv,2517,264460
4,monthly_overview.csv,18,1057
5,routing_ab.csv,30,1931
6,settlement_performance.csv,72,6023


Cross-checks OK — txn-weighted SR 83.69% | settlement breach 28.43%
